In [2]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import random
import json
from google.colab import files
SEED = 900729
dir_path = './data'
train_path = './data/Training'
test_path = './data/Testing'
uploaded=files.upload()


Saving kaggle.json to kaggle (2).json


In [3]:
!ls
with open("kaggle.json") as f:
    kaggle_creds = json.load(f)

username = kaggle_creds["username"]
key = kaggle_creds["key"]

os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key

 data  'kaggle (1).json'  'kaggle (2).json'   kaggle.json   sample_data


In [4]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()



api.dataset_download_files("masoudnickparvar/brain-tumor-mri-dataset", path="./data", unzip=True)




Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset


In [5]:
def train_df(tr_path):
    classes, class_paths = zip(*[(label, os.path.join(tr_path, label, image))
                                 for label in os.listdir(tr_path) if os.path.isdir(os.path.join(tr_path, label))
                                 for image in os.listdir(os.path.join(tr_path, label))])

    tr_df = pd.DataFrame({'Class Path': class_paths, 'Class': classes})
    return tr_df

In [6]:
def test_df(ts_path):
    classes, class_paths = zip(*[(label, os.path.join(ts_path, label, image))
                                 for label in os.listdir(ts_path) if os.path.isdir(os.path.join(ts_path, label))
                                 for image in os.listdir(os.path.join(ts_path, label))])

    ts_df = pd.DataFrame({'Class Path': class_paths, 'Class': classes})
    return ts_df

In [7]:
tr_orginal_df = train_df(train_path)
ts_orginal_df = test_df(test_path)
ts_orginal_df

,Class Path,Class
0,./data/Testing/meningioma/Te-me_0156.jpg,meningioma
1,./data/Testing/meningioma/Te-me_0026.jpg,meningioma
2,./data/Testing/meningioma/Te-me_0094.jpg,meningioma
3,./data/Testing/meningioma/Te-me_0075.jpg,meningioma
4,./data/Testing/meningioma/Te-me_0167.jpg,meningioma
...,...,...
1306,./data/Testing/glioma/Te-gl_0108.jpg,glioma
1307,./data/Testing/glioma/Te-gl_0024.jpg,glioma
1308,./data/Testing/glioma/Te-glTr_0007.jpg,glioma
1309,./data/Testing/glioma/Te-gl_0235.jpg,glioma


In [8]:
tr_orginal_df

,Class Path,Class
0,./data/Training/meningioma/Tr-me_0322.jpg,meningioma
1,./data/Training/meningioma/Tr-me_1304.jpg,meningioma
2,./data/Training/meningioma/Tr-me_0840.jpg,meningioma
3,./data/Training/meningioma/Tr-me_1295.jpg,meningioma
4,./data/Training/meningioma/Tr-me_0451.jpg,meningioma
...,...,...
5707,./data/Training/glioma/Tr-gl_0171.jpg,glioma
5708,./data/Training/glioma/Tr-gl_1118.jpg,glioma
5709,./data/Training/glioma/Tr-gl_0329.jpg,glioma
5710,./data/Training/glioma/Tr-gl_0592.jpg,glioma


In [9]:
val_df, tr_df = train_test_split(tr_orginal_df, train_size=0.25, random_state=SEED, stratify=tr_orginal_df['Class'])

validation_path = './data/Validation'
Path(validation_path).mkdir(exist_ok=True)

classes = ['glioma', 'meningioma', 'notumor', 'pituitary']
for class_name in classes:
    class_validation_path = Path(validation_path) / class_name
    class_validation_path.mkdir(exist_ok=True)

errors = []

for idx, row in val_df.iterrows():
    try:
        source_path = Path(row['Class Path'])

        filename = source_path.name
        class_name = row['Class']
        destination_path = Path(validation_path) / class_name / filename

        shutil.move(str(source_path), str(destination_path))

    except Exception as e:
        errors.append({
            'file': row['Class Path'],
            'class': row['Class'],
            'error': str(e)
        })


if errors:
    print(f"Errors: {len(errors)}")
    for error in errors:
        print(f"  - {error['class']}: {Path(error['file']).name} - {error['error']}")

val_df_updated = val_df.copy()
val_df_updated['Class Path'] = val_df_updated.apply(
    lambda row: str(Path(validation_path) / row['Class'] / Path(row['Class Path']).name),
    axis=1
)

tr_df_updated = train_df(train_path)

val_df = val_df_updated
tr_df = tr_df_updated

val_df

,Class Path,Class
312,data/Validation/meningioma/Tr-me_0680.jpg,meningioma
2060,data/Validation/notumor/Tr-no_1069.jpg,notumor
5526,data/Validation/glioma/Tr-glTr_0007.jpg,glioma
5546,data/Validation/glioma/Tr-gl_0105.jpg,glioma
4935,data/Validation/glioma/Tr-gl_1046.jpg,glioma
...,...,...
3281,data/Validation/pituitary/Tr-pi_0379.jpg,pituitary
847,data/Validation/meningioma/Tr-me_1066.jpg,meningioma
86,data/Validation/meningioma/Tr-me_0122.jpg,meningioma
1220,data/Validation/meningioma/Tr-me_1251.jpg,meningioma


In [10]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


transform_tr = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
transform_ts=transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
transform_val =transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = datasets.ImageFolder(root=train_path, transform=transform_tr)
test_dataset  = datasets.ImageFolder(root=test_path, transform=transform_ts)
val_dataset   = datasets.ImageFolder(root=validation_path, transform=transform_val)

# Reduced batch size to mitigate OutOfMemoryError
batch_size = 16 # Reduced from 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(train_dataset.classes)

['glioma', 'meningioma', 'notumor', 'pituitary']


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

class UpSampleConcat(nn.Module):
    def __init__(self, scale_factor=2):
        super().__init__()
        self.scale_factor = scale_factor

    def forward(self, x1, x2):
        # Upsample x1 to x2 size and concatenate
        x1_up = F.interpolate(x1, size=x2.shape[2:], mode='bilinear', align_corners=True)
        return torch.cat([x1_up, x2], dim=1)

class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=3, out_channels=4, base_filters=16):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.base_filters = base_filters

        # Encoder path (x0_0, x1_0, x2_0, x3_0, x4_0)
        self.conv0_0 = ConvBlock(in_channels, base_filters)
        self.conv1_0 = ConvBlock(base_filters, base_filters * 2)
        self.conv2_0 = ConvBlock(base_filters * 2, base_filters * 4)
        self.conv3_0 = ConvBlock(base_filters * 4, base_filters * 8)
        self.conv4_0 = ConvBlock(base_filters * 8, base_filters * 16)

        self.pool = nn.MaxPool2d(2, 2)

        # Decoder path with nested skip connections
        # Corrected input channels for convolutional blocks in the decoder
        self.up1_0_1 = UpSampleConcat()
        self.conv0_1 = ConvBlock(base_filters + base_filters * 2, base_filters)

        self.up2_0_1 = UpSampleConcat()
        self.conv1_1 = ConvBlock(base_filters * 2 + base_filters * 4, base_filters * 2)

        self.up3_0_1 = UpSampleConcat()
        self.conv2_1 = ConvBlock(base_filters * 4 + base_filters * 8, base_filters * 4)

        self.up4_0_1 = UpSampleConcat()
        self.conv3_1 = ConvBlock(base_filters * 8 + base_filters * 16, base_filters * 8)

        self.up1_1_2 = UpSampleConcat()
        self.conv0_2 = ConvBlock(base_filters + base_filters + base_filters * 2, base_filters) # x0_0, x0_1, upsampled x1_1

        self.up2_1_2 = UpSampleConcat()
        self.conv1_2 = ConvBlock(base_filters * 2 + base_filters * 2 + base_filters * 4, base_filters * 2) # x1_0, x1_1, upsampled x2_1

        self.up3_1_2 = UpSampleConcat()
        self.conv2_2 = ConvBlock(base_filters * 4 + base_filters * 4 + base_filters * 8, base_filters * 4) # x2_0, x2_1, upsampled x3_1

        self.up1_2_3 = UpSampleConcat()
        self.conv0_3 = ConvBlock(base_filters + base_filters + base_filters + base_filters * 2, base_filters) # x0_0, x0_1, x0_2, upsampled x1_2

        self.up2_2_3 = UpSampleConcat()
        self.conv1_3 = ConvBlock(base_filters * 2 + base_filters * 2 + base_filters * 2 + base_filters * 4, base_filters * 2) # x1_0, x1_1, x1_2, upsampled x2_2

        self.up1_3_4 = UpSampleConcat()
        self.conv0_4 = ConvBlock(base_filters + base_filters + base_filters + base_filters + base_filters * 2, base_filters) # x0_0, x0_1, x0_2, x0_3, upsampled x1_3


        self.final = nn.Conv2d(base_filters, out_channels, kernel_size=1)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x2_0 = self.conv2_0(self.pool(x1_0))
        x3_0 = self.conv3_0(self.pool(x2_0))
        x4_0 = self.conv4_0(self.pool(x3_0))

        x3_1 = self.conv3_1(self.up4_0_1(x4_0, x3_0))
        x2_1 = self.conv2_1(self.up3_0_1(x3_1, x2_0))
        x1_1 = self.conv1_1(self.up2_0_1(x2_1, x1_0))
        x0_1 = self.conv0_1(self.up1_0_1(x1_1, x0_0))

        x2_2 = self.conv2_2(self.up3_1_2(x3_1, torch.cat([x2_0, x2_1], dim=1))) # Concatenate x2_0 and x2_1 for skip connection
        x1_2 = self.conv1_2(self.up2_1_2(x2_2, torch.cat([x1_0, x1_1], dim=1))) # Concatenate x1_0 and x1_1 for skip connection
        x0_2 = self.conv0_2(self.up1_1_2(x1_2, torch.cat([x0_0, x0_1], dim=1))) # Concatenate x0_0 and x0_1 for skip connection

        x1_3 = self.conv1_3(self.up2_2_3(x2_2, torch.cat([x1_0, x1_1, x1_2], dim=1))) # Concatenate x1_0, x1_1, x1_2 for skip connection
        x0_3 = self.conv0_3(self.up1_2_3(x1_3, torch.cat([x0_0, x0_1, x0_2], dim=1))) # Concatenate x0_0, x0_1, x0_2 for skip connection

        x0_4 = self.conv0_4(self.up1_3_4(x1_3, torch.cat([x0_0, x0_1, x0_2, x0_3], dim=1))) # Concatenate x0_0, x0_1, x0_2, x0_3 for skip connection


        out = self.final(x0_4)
        return out

In [12]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0 # Reset patience
        else:
            self.counter += 1 # Increase if no improvement
        return self.counter >= self.patience # True = stop

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UNetPlusPlusForClassification(nn.Module):
    def __init__(self, unetpp_model, num_classes):
        super().__init__()
        self.unetpp = unetpp_model
        # Modify the final layer for classification
        # We'll remove the original final convolutional layer and add a new classification head
        self.classification_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # Global Average Pooling
            nn.Flatten(),
            nn.Linear(self.unetpp.base_filters, num_classes) # Linear layer for classification
        )

    def forward(self, x):
        # Pass through the UNet++ encoder and decoder up to the last feature map before the original final layer
        x0_0 = self.unetpp.conv0_0(x)
        x1_0 = self.unetpp.conv1_0(self.unetpp.pool(x0_0))
        x2_0 = self.unetpp.conv2_0(self.unetpp.pool(x1_0))
        x3_0 = self.unetpp.conv3_0(self.unetpp.pool(x2_0))
        x4_0 = self.unetpp.conv4_0(self.unetpp.pool(x3_0))

        x3_1 = self.unetpp.conv3_1(self.unetpp.up4_0_1(x4_0, x3_0))
        x2_1 = self.unetpp.conv2_1(self.unetpp.up3_0_1(x3_1, x2_0))
        x1_1 = self.unetpp.conv1_1(self.unetpp.up2_0_1(x2_1, x1_0))
        x0_1 = self.unetpp.conv0_1(self.unetpp.up1_0_1(x1_1, x0_0))

        x2_2 = self.unetpp.conv2_2(self.unetpp.up3_1_2(x3_1, torch.cat([x2_0, x2_1], dim=1)))
        x1_2 = self.unetpp.conv1_2(self.unetpp.up2_1_2(x2_2, torch.cat([x1_0, x1_1], dim=1)))
        x0_2 = self.unetpp.conv0_2(self.unetpp.up1_1_2(x1_2, torch.cat([x0_0, x0_1], dim=1)))

        x1_3 = self.unetpp.conv1_3(self.unetpp.up2_2_3(x2_2, torch.cat([x1_0, x1_1, x1_2], dim=1)))
        x0_3 = self.unetpp.conv0_3(self.unetpp.up1_2_3(x1_3, torch.cat([x0_0, x0_1, x0_2], dim=1)))

        x0_4 = self.unetpp.conv0_4(self.unetpp.up1_3_4(x1_3, torch.cat([x0_0, x0_1, x0_2, x0_3], dim=1)))

        # Pass the final feature map through the classification head
        out = self.classification_head(x0_4)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the original UNetPlusPlus model (without the final segmentation layer)
# and then wrap it with the classification head
unetpp_base = UNetPlusPlus(in_channels=3, out_channels=4, base_filters=32) # out_channels here doesn't matter as we replace the final layer
model = UNetPlusPlusForClassification(unetpp_base, num_classes=len(train_dataset.classes))
model.to(device)

print(f"Using device: {device}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
early_stopping = EarlyStopping(patience=5, min_delta=0.001)
epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images) # Outputs shape: [batch, num_classes]

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()


    epoch_loss = running_loss / len(train_loader.dataset)
    train_accuracy = 100 * correct_train / total_train
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader.dataset)
    val_accuracy = 100 * correct_val / total_val
    print(f"Epoch [{epoch+1}/{epochs}], Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    if early_stopping(val_loss):
        print("Early stopping triggered!")
        break

print("Training finished.")

Using device: cuda
Epoch [1/30], Train Loss: 1.0271, Train Accuracy: 62.14%
Epoch [1/30], Validation Loss: 0.8710, Validation Accuracy: 69.96%
Epoch [2/30], Train Loss: 0.8344, Train Accuracy: 71.92%
Epoch [2/30], Validation Loss: 0.7629, Validation Accuracy: 72.41%
Epoch [3/30], Train Loss: 0.6917, Train Accuracy: 78.45%
Epoch [3/30], Validation Loss: 0.4881, Validation Accuracy: 83.47%
Epoch [4/30], Train Loss: 0.6007, Train Accuracy: 81.96%
Epoch [4/30], Validation Loss: 0.4965, Validation Accuracy: 84.38%
Epoch [5/30], Train Loss: 0.5635, Train Accuracy: 82.19%
Epoch [5/30], Validation Loss: 0.4310, Validation Accuracy: 85.85%
Epoch [6/30], Train Loss: 0.4914, Train Accuracy: 84.92%
Epoch [6/30], Validation Loss: 0.4561, Validation Accuracy: 84.59%
Epoch [7/30], Train Loss: 0.4598, Train Accuracy: 85.62%
Epoch [7/30], Validation Loss: 0.3736, Validation Accuracy: 87.96%
Epoch [8/30], Train Loss: 0.4265, Train Accuracy: 86.90%
Epoch [8/30], Validation Loss: 0.3337, Validation Accura

In [1]:
torch.save(model.state_dict(), 'model.pth')
files.download('model.pth')



NameError: name 'torch' is not defined

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()  # Set the model to evaluation mode
test_loss = 0.0
correct_test = 0
total_test = 0
all_labels = []
all_predicted = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

avg_test_loss = test_loss / len(test_loader.dataset)
test_accuracy = 100 * correct_test / total_test

print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

# Display Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_predicted, target_names=train_dataset.classes))

# Display Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_predicted)
print(cm)

# Optional: Visualize Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
!pip install grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import random
import cv2
from PIL import Image
target_layers = [model.unetpp.conv4_0]  # Ostatnia warstwa konwolucyjna
cam = GradCAM(model=model, target_layers=target_layers)

# Wybierz 8 losowych próbek
indices = torch.randperm(len(test_dataset))[:8]

# Pobieramy próbki
samples = [test_dataset[i] for i in indices]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, (img, true_label) in zip(axes.flatten(), samples):
    img = img.to(device).unsqueeze(0)
    # Predykcja
    with torch.no_grad():
        outputs = model(img)
        predicted_class = outputs.argmax(dim=1).item()
        probabilities = F.softmax(outputs, dim=1)[0]

    # Grad-CAM
    targets = [ClassifierOutputTarget(predicted_class)]
    grayscale_cam = cam(input_tensor=img, targets=targets)[0, :]
    visualization = show_cam_on_image(img, grayscale_cam, use_rgb=True)

    # Wyświetlanie
    ax.imshow(visualization)
    ax.set_title(
        f"Pred: {test_dataset.classes[predicted_class]}\nTrue: {test_dataset.classes[true_label]}\nConf: {probabilities[predicted_class]:.3f}",
        fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()